## INIT

In [0]:
import sys
import os
sys.path.append(os.path.abspath('../../'))
from utils.logger_utils import get_project_logger

logger = get_project_logger("Gold_fact_public_vehicles")
logger.info("--- Starting Gold Layer: Creating fact_public_vehicles ---")

## Creating fact view

In [0]:
try:
    logger.info("Step 1/4: Running SQL to create fact_public_vehicles view")
    
    sql_query = """
    CREATE OR REPLACE VIEW transport_lakehouse.gold.fact_public_vehicles AS
    SELECT
    p.car_num,
    m.manufacturer_key,
    vt.vehicle_type_key,
    c.color_key,
    P.commercial_nm,
    p.total_weight,
    p.capacity_amount,
    p.prod_year,
    p.test_valid_until_dt,
    p.canceled_cd,
    p.canceled_nm,
    p.canceled_dt,
    1 AS vehicle_count

    FROM transport_lakehouse.silver.silver_vehicles_public p
    LEFT JOIN transport_lakehouse.gold.dim_manufacturer m
    ON CAST(p.manufacturer_cd AS STRING)=CAST(m.manufacturer_cd AS STRING)
    LEFT JOIN transport_lakehouse.gold.dim_vehicle_type vt
    ON CAST(p.vehicle_type_cd AS STRING)=CAST(vt.vehicle_type_cd AS STRING)
    AND p.vehicle_type_nm = vt.vehicle_type_nm
    LEFT JOIN transport_lakehouse.gold.dim_color c
    ON CAST(p.color_cd AS STRING)=CAST(c.color_cd AS STRING)
    ORDER BY p.canceled_cd
     """
    
    spark.sql(sql_query)
    logger.info("Step 1/4: View created successfully.")

except Exception as e:
    logger.error(f"Failed to create Gold Dimension: {str(e)}")
    raise e


## Missing keys check

In [0]:
logger.info("Step 2/4: Starting missing keys check...")

df_check = spark.sql("""
SELECT
  SUM(CASE WHEN manufacturer_key IS NULL THEN 1 ELSE 0 END) AS missing_manufacturer,
  SUM(CASE WHEN vehicle_type_key IS NULL THEN 1 ELSE 0 END) AS missing_vehicle_type,
  SUM(CASE WHEN color_key IS NULL THEN 1 ELSE 0 END) AS missing_color
FROM transport_lakehouse.gold.fact_public_vehicles
""")

results = df_check.collect()[0]

missing_report = (
    f"Missing Keys Summary: "
    f"Manufacturers: {results['missing_manufacturer']}, "
    f"Colors: {results['missing_color']}, "
    f"vehicle_types: {results['missing_vehicle_type']}"
)

if (results['missing_manufacturer'] + results['missing_color'] + 
    results['missing_vehicle_type']) > 0:
    logger.warning(f"Step 2/4: Finished with MISSING KEYS! {missing_report}")
else:
    logger.info(f"Step 2/4: Finished missing keys check successfully. All keys are mapped. {missing_report}")


## Duplication check

In [0]:
logger.info("Step 3/4: Starting Duplication check...")

df_check = spark.sql("""
SELECT COUNT(*) AS rows, COUNT(DISTINCT car_num) AS distinct_cars
FROM transport_lakehouse.gold.fact_private_vehicles
""")

results = df_check.collect()[0]

if results['rows'] == results['distinct_cars']:
    logger.info(f"Step 2/4: Finished duplication check successfully. All rows are unique.")
else:
    logger.warning(f"Step 2/4: Finished with DUPLICATE ROWS!")

## Key null ratio check

In [0]:
logger.info("Step 4/4: Starting Key null ratio check...")

df_check = spark.sql("""
SELECT
  CAST(AVG(CASE WHEN manufacturer_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS manufacturer_key_fill_rate,
  CAST(AVG(CASE WHEN vehicle_type_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS vehicle_type_key_fill_rate,
  CAST(AVG(CASE WHEN color_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS  color_key_fill_rate
FROM transport_lakehouse.gold.fact_public_vehicles
""")

results = df_check.collect()[0]

missing_report = (
    f"Missing Keys Summary: "
    f"Manufacturers: {results['manufacturer_key_fill_rate']}, "
    f"Colors: {results['color_key_fill_rate']}, "
    f"Fuel: {results['vehicle_type_key_fill_rate']}"
)

if (results['manufacturer_key_fill_rate'] < 1 or 
    results['color_key_fill_rate'] < 1 or 
    results['vehicle_type_key_fill_rate'] < 1):
    logger.warning(f"Step 4/4: Finished with MISSING KEYS! {missing_report}")
else:
    logger.info(f"Step 4/4: Finished  Key null ratio successfully. All keys are mapped. {missing_report}")
    logger.info("--- Gold fact_public_vehicles Process Completed ---")
